# **Tugas 2 - Kata Unik pada Dataset**

---



## **1. Install library terlebih dahulu**



In [1]:
pip install pandas openpyxl numpy scikit-learn

## **2. Import library**

---

Kode tersebut digunakan untuk menyiapkan berbagai library dalam proses **pengolahan data dan klasifikasi teks menggunakan Machine Learning**. `pandas` dan `numpy` digunakan untuk mengolah data, sedangkan `re` untuk membersihkan atau memproses teks. `csr_matrix` digunakan untuk mengelola data dalam bentuk sparse matrix. `train_test_split` digunakan untuk membagi data menjadi data latih dan data uji, sementara `TfidfVectorizer` mengubah teks menjadi nilai numerik berdasarkan bobot TF-IDF. `PCA` digunakan untuk mengurangi jumlah fitur, dan `LogisticRegression` digunakan sebagai algoritma klasifikasi. Terakhir, `accuracy_score`, `classification_report`, dan `confusion_matrix` digunakan untuk **mengukur dan mengevaluasi kinerja model**.



In [2]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## **3. Membaca dataset**

In [3]:
df = pd.read_excel("dataset_detik_200_berita.xlsx")

df.head()

,id,isi_berita,label
0,1,Pacuan kuda Indonesia mulai diarahkan menuju p...,sport
1,2,"Jelang Asian Games 2026, dukungan untuk 432 at...",sport
2,3,Nova Widianto mulai menangani tim ganda campur...,sport
3,4,Timnas basket putra Indonesia jadi rombongan p...,sport
4,5,"Bagi pelari urban, lari kini bukan lagi sekada...",sport


### **Cek jumlah data:**

In [4]:
print("Jumlah data :", len(df))
print("\nNama kolom:")
print(df.columns)

print("\nJumlah data per label:")
print(df["label"].value_counts())

Jumlah data : 200

Nama kolom:
Index(['id', 'isi_berita', 'label'], dtype='object')

Jumlah data per label:
label
sport      100
finance    100
Name: count, dtype: int64


## **4. Mengubah label menjadi numerik**

In [5]:
df["label_num"] = df["label"].map({
    "sport": 1,
    "finance": 0
})

df[["id", "label", "label_num"]].head()

,id,label,label_num
0,1,sport,1
1,2,sport,1
2,3,sport,1
3,4,sport,1
4,5,sport,1


In [6]:
print(df["label_num"].value_counts())

label_num
1    100
0    100
Name: count, dtype: int64


## **5. Menghitung jumlah kata semua berita**

In [7]:
df["jumlah_kata_asli"] = df["isi_berita"].astype(str).apply(
    lambda x: len(x.split())
)

df[["id", "jumlah_kata_asli"]].head()

total_kata = df["jumlah_kata_asli"].sum()

print("Total seluruh kata :", total_kata)

print(df["jumlah_kata_asli"].describe())

Total seluruh kata : 62652
count     200.000000
mean      313.260000
std       132.649053
min        41.000000
25%       241.250000
50%       293.000000
75%       352.500000
max      1033.000000
Name: jumlah_kata_asli, dtype: float64


### **Melihat jumlah kata pada setiap berita**

In [8]:
df[[
    "id",
    "label",
    "jumlah_kata_asli"
]]

,id,label,jumlah_kata_asli
0,1,sport,396
1,2,sport,291
2,3,sport,331
3,4,sport,315
4,5,sport,546
...,...,...,...
195,196,finance,546
196,197,finance,493
197,198,finance,217
198,199,finance,296


## **6. Kamus kata tidak baku**

In [9]:
kamus_tidak_baku = {
    "gak": "tidak",
    "nggak": "tidak",
    "ga": "tidak",
    "enggak": "tidak",
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "kalo": "kalau",
    "kalok": "kalau",
    "aja": "saja",
    "udah": "sudah",
    "sdh": "sudah",
    "blm": "belum",
    "tdk": "tidak",
    "dr": "dari",
    "dlm": "dalam",
    "jd": "jadi",
    "bgt": "banget",
    "tp": "tetapi",
    "tapi": "tetapi",
    "karna": "karena",
    "trus": "terus",
    "kmrn": "kemarin",
    "dpt": "dapat",
    "hrs": "harus",
    "sm": "sama",
    "sy": "saya"
}

## **7. Kamus bahasa asing**

In [10]:
kamus_asing = {

    # SPORT
    "rider": "pembalap",
    "race": "balapan",
    "racing": "balap",
    "team": "tim",
    "coach": "pelatih",
    "player": "pemain",
    "match": "pertandingan",
    "winner": "pemenang",
    "season": "musim",
    "training": "latihan",
    "game": "pertandingan",
    "games": "pertandingan",
    "manager": "manajer",
    "champion": "juara",
    "championship": "kejuaraan",
    "league": "liga",
    "score": "skor",
    "goal": "gol",
    "final": "final",

    # FINANCE
    "finance": "keuangan",
    "financial": "keuangan",
    "market": "pasar",
    "stock": "saham",
    "stocks": "saham",
    "sale": "penjualan",
    "price": "harga",
    "business": "bisnis",
    "company": "perusahaan",
    "investment": "investasi",
    "investor": "investor",
    "banking": "perbankan",
    "bank": "bank",
    "economy": "ekonomi",
    "economic": "ekonomi",
    "growth": "pertumbuhan",
    "profit": "keuntungan",
    "loss": "kerugian",
    "revenue": "pendapatan"
}

## **8. Fungsi preprocessing**

---

Fungsi `preprocessing()` digunakan untuk **membersihkan dan menyiapkan data teks sebelum diproses oleh model Machine Learning**. Prosesnya dimulai dengan mengubah teks menjadi string dan melakukan *case folding* dengan mengubah semua huruf menjadi lowercase. Selanjutnya, fungsi menghapus URL, email, angka, tanda baca, simbol, emoticon, serta spasi berlebih. Teks kemudian **ditokenisasi** menjadi kata-kata, lalu kata tidak baku diperbaiki menggunakan `kamus_tidak_baku` dan kata asing diterjemahkan menggunakan `kamus_asing`. Setelah seluruh proses selesai, kata-kata tersebut digabung kembali menjadi teks yang bersih dan siap digunakan untuk tahap analisis selanjutnya.


In [11]:
def preprocessing(text):

    # Pastikan berupa string
    text = str(text)

    # =========================
    # CASE FOLDING
    # =========================
    text = text.lower()


    # =========================
    # HAPUS URL
    # =========================
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ',
        text
    )


    # =========================
    # HAPUS EMAIL
    # =========================
    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )


    # =========================
    # HAPUS ANGKA
    # =========================
    text = re.sub(
        r'\d+',
        ' ',
        text
    )


    # =========================
    # HAPUS TANDA BACA,
    # SIMBOL DAN EMOTICON
    # =========================

    text = re.sub(
        r'[^a-zA-ZÀ-ÿ\s]',
        ' ',
        text
    )


    # =========================
    # HAPUS SPASI BERLEBIH
    # =========================

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()


    # =========================
    # TOKENISASI
    # =========================

    kata = text.split()


    hasil = []

    for token in kata:

        # Membakukan kata
        if token in kamus_tidak_baku:
            token = kamus_tidak_baku[token]


        # Bahasa asing -> Indonesia
        if token in kamus_asing:
            token = kamus_asing[token]


        hasil.append(token)


    return " ".join(hasil)

## **9. Penerapan preprocessing ke semua berita**

In [12]:
df["berita_clean"] = df["isi_berita"].apply(preprocessing)

df[[
    "id",
    "isi_berita",
    "berita_clean"
]].head()

,id,isi_berita,berita_clean
0,1,Pacuan kuda Indonesia mulai diarahkan menuju p...,pacuan kuda indonesia mulai diarahkan menuju p...
1,2,"Jelang Asian Games 2026, dukungan untuk 432 at...",jelang asian pertandingan dukungan untuk atlet...
2,3,Nova Widianto mulai menangani tim ganda campur...,nova widianto mulai menangani tim ganda campur...
3,4,Timnas basket putra Indonesia jadi rombongan p...,timnas basket putra indonesia jadi rombongan p...
4,5,"Bagi pelari urban, lari kini bukan lagi sekada...",bagi pelari urban lari kini bukan lagi sekadar...


## **10. Hitung jumlah kata setelah preprocessing**

In [13]:
df["jumlah_kata_clean"] = df["berita_clean"].apply(
    lambda x: len(x.split())
)

df[[
    "id",
    "jumlah_kata_asli",
    "jumlah_kata_clean"
]].head()

,id,jumlah_kata_asli,jumlah_kata_clean
0,1,396,387
1,2,291,286
2,3,331,325
3,4,315,311
4,5,546,541


### **Total Sebelum dan Setelah Preprocessing**

In [53]:
print(
    "Total kata sebelum preprocessing :",
    df["jumlah_kata_asli"].sum()
)

print(
    "Total kata setelah preprocessing :",
    df["jumlah_kata_clean"].sum()
)

Total kata sebelum preprocessing : 62652
Total kata setelah preprocessing : 61162


## **11. Ekstrak seluruh kata unik**

In [15]:
semua_kata = []

for berita in df["berita_clean"]:
    semua_kata.extend(
        berita.split()
    )

kata_unik = sorted(
    set(semua_kata)
)

print(
    "Jumlah seluruh kata:",
    len(semua_kata)
)

print(
    "Jumlah kata unik:",
    len(kata_unik)
)

Jumlah seluruh kata: 61162
Jumlah kata unik: 7019


In [16]:
kata_unik[:100]

['a',
 'aadi',
 'aan',
 'abal',
 'abang',
 'abdi',
 'abdul',
 'abraham',
 'absen',
 'absennya',
 'absorber',
 'abu',
 'abullah',
 'acara',
 'access',
 'account',
 'acd',
 'aceh',
 'acid',
 'acosta',
 'activ',
 'activation',
 'activities',
 'acuan',
 'ada',
 'adain',
 'adalah',
 'adanya',
 'adaptasi',
 'adapun',
 'adhitya',
 'adi',
 'adianto',
 'adik',
 'adil',
 'administrasi',
 'administratif',
 'adna',
 'adopsi',
 'adp',
 'adriatik',
 'ads',
 'adu',
 'aduan',
 'advisory',
 'advokat',
 'aerodinamika',
 'aeronautika',
 'aesi',
 'af',
 'aff',
 'afrianto',
 'afrika',
 'ag',
 'agak',
 'agama',
 'agar',
 'agenda',
 'agent',
 'agraria',
 'agreement',
 'agregat',
 'agresif',
 'agresivitas',
 'agrinas',
 'agung',
 'agus',
 'agusman',
 'agustus',
 'agustusan',
 'ahi',
 'ahmad',
 'ahmed',
 'ahren',
 'ahsanurrohim',
 'ai',
 'aichi',
 'air',
 'airbus',
 'airin',
 'airlangga',
 'airlines',
 'airmen',
 'airnav',
 'airport',
 'airports',
 'ajaib',
 'ajakan',
 'ajang',
 'ajaran',
 'ajir',
 'akademik',

In [17]:
df_kata_unik = pd.DataFrame({
    "kata_unik": kata_unik
})

df_kata_unik.head(20)

,kata_unik
0,a
1,aadi
2,aan
3,abal
4,abang
5,abdi
6,abdul
7,abraham
8,absen
9,absennya


In [18]:
df_kata_unik.to_excel(
    "kata_unik.xlsx",
    index=False
)

## **12. Membagi data 160 training dan 40 testing**

In [19]:
X_train_text, X_test_text, y_train, y_test = train_test_split(

    df["berita_clean"],
    df["label_num"],

    test_size=40,

    random_state=42,

    stratify=df["label_num"]
)

In [20]:
print(
    "Jumlah training:",
    len(X_train_text)
)

print(
    "Jumlah testing:",
    len(X_test_text)
)

Jumlah training: 160
Jumlah testing: 40


In [21]:
print("TRAINING")
print(y_train.value_counts())

print("\nTESTING")
print(y_test.value_counts())

TRAINING
label_num
0    80
1    80
Name: count, dtype: int64

TESTING
label_num
0    20
1    20
Name: count, dtype: int64


## **13. Representasi TF-IDF**

In [22]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_test_tfidf = tfidf.transform(
    X_test_text
)

In [23]:
nama_fitur = tfidf.get_feature_names_out()

print(
    "Jumlah fitur TF-IDF:",
    len(nama_fitur)
)

Jumlah fitur TF-IDF: 3038


## **14. Melihat matriks TF-IDF**

In [24]:
tfidf_train_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_train_df.head()

,abdul,absen,abu,acara,acd,aceh,acosta,ada,adalah,adanya,...,youtube,yoy,yudha,yudhi,yusrian,zaman,zapp,zona,zulhas,zulkifli
0,0.0,0.0,0.0,0.028231,0.0,0.0,0.0,0.038789,0.014256,0.0,...,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.055677,0.092084,0.0,...,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.144931,0.000000,0.0,...,0.0,0.0,0.0,0.03154,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.021976,0.048461,0.0,...,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
tfidf_train_df["label"] = y_train.reset_index(
    drop=True
)

tfidf_train_df.head()

,abdul,absen,abu,acara,acd,aceh,acosta,ada,adalah,adanya,...,yoy,yudha,yudhi,yusrian,zaman,zapp,zona,zulhas,zulkifli,label
0,0.0,0.0,0.0,0.028231,0.0,0.0,0.0,0.038789,0.014256,0.0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.055677,0.092084,0.0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.144931,0.000000,0.0,...,0.0,0.0,0.03154,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.021976,0.048461,0.0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1


## **15. Simpan data TF-IDF**

In [26]:
tfidf_train_df.to_excel(
    "data_tfidf_training.xlsx",
    index=False
)

In [27]:
tfidf_test_df = pd.DataFrame(
    X_test_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_test_df["label"] = y_test.reset_index(
    drop=True
)

tfidf_test_df.to_excel(
    "data_tfidf_testing.xlsx",
    index=False
)

In [28]:
features = np.array(
    tfidf.get_feature_names_out()
)

jumlah_kelas = y_train.nunique()

class_space_density = np.zeros(
    len(features)
)

y_train_array = y_train.to_numpy()

In [29]:
for kelas in sorted(
    y_train.unique()
):

    mask = (
        y_train_array == kelas
    )

    X_class = X_train_tfidf[
        mask
    ]


    # Berapa dokumen kelas tersebut
    # mengandung sebuah kata

    document_frequency_class = (
        (X_class > 0)
        .sum(axis=0)
        .A1
    )


    jumlah_dokumen_class = (
        X_class.shape[0]
    )


    class_density = (
        document_frequency_class
        /
        jumlah_dokumen_class
    )


    class_space_density += (
        class_density
    )

In [30]:
epsilon = 1e-12

icsdf = np.log(

    (jumlah_kelas + epsilon)

    /

    (class_space_density + epsilon)
)

In [31]:
df_icsdf = pd.DataFrame({

    "kata": features,

    "ICSDF": icsdf
})

df_icsdf.sort_values(
    "ICSDF",
    ascending=False
).head(20)

,kata,ICSDF
10,adaptasi,4.382027
3037,zulkifli,4.382027
3036,zulhas,4.382027
1,absen,4.382027
3033,zaman,4.382027
12,adi,4.382027
3030,yudha,4.382027
5,aceh,4.382027
3013,wti,4.382027
14,adu,4.382027


## **16. TF-IDF × ICSDF**

In [32]:
X_train_icsdf = X_train_tfidf.multiply(
    icsdf
)

X_test_icsdf = X_test_tfidf.multiply(
    icsdf
)

In [33]:
X_train_icsdf = csr_matrix(
    X_train_icsdf
)

X_test_icsdf = csr_matrix(
    X_test_icsdf
)

## **17. Menentukan kata paling penting**

---
 kode di bawah digunakan untuk mencari **kata-kata yang memiliki nilai atau bobot fitur tertinggi** berdasarkan data TF-IDF pada `X_train_icsdf`. Nilai rata-rata setiap fitur dihitung menggunakan `mean(axis=0)`, kemudian `np.argsort()` digunakan untuk mengurutkan fitur dari skor terendah hingga tertinggi dan mengambil **100 kata dengan skor tertinggi**. Kata-kata tersebut kemudian disimpan bersama skornya dalam DataFrame `df_kata_penting`, sehingga dapat diketahui kata mana yang paling dominan dalam dataset.

In [34]:
skor_fitur = np.asarray(
    X_train_icsdf.mean(
        axis=0
    )
).ravel()

In [35]:
TOP_K = 100

In [36]:
top_index = np.argsort(
    skor_fitur
)[::-1][:TOP_K]

In [37]:
kata_penting = features[
    top_index
]

kata_penting[:30]

array(['saham', 'bandara', 'marquez', 'motogp', 'marc', 'balapan',
       'purbaya', 'set', 'aset', 'padel', 'rp', 'aragon', 'olahraga',
       'anggaran', 'emas', 'bezzecchi', 'harga', 'martin', 'atlet',
       'beras', 'saya', 'bbm', 'pertandingan', 'ihsg', 'perdagangan',
       'antonelli', 'masker', 'pasar', 'program', 'triliun'], dtype=object)

In [38]:
df_kata_penting = pd.DataFrame({

    "kata": kata_penting,

    "skor": skor_fitur[
        top_index
    ]
})

df_kata_penting.head(30)

,kata,skor
0,saham,0.058117
1,bandara,0.058049
2,marquez,0.056400
3,motogp,0.051358
4,marc,0.050835
5,balapan,0.050778
6,purbaya,0.049831
7,set,0.045378
8,aset,0.042438
9,padel,0.039189


## **18. Reduksi menjadi 100 fitur ICSDF**

In [39]:
X_train_selected = X_train_icsdf[
    :,
    top_index
]

X_test_selected = X_test_icsdf[
    :,
    top_index
]

In [40]:
print(
    "Sebelum seleksi:",
    X_train_tfidf.shape
)

print(
    "Sesudah ICSDF:",
    X_train_selected.shape
)

Sebelum seleksi: (160, 3038)
Sesudah ICSDF: (160, 100)


## **19. Tabel Hasil ICSDF**

In [41]:
icsdf_train_df = pd.DataFrame(

    X_train_selected.toarray(),

    columns=kata_penting
)

icsdf_train_df["label"] = (

    y_train.reset_index(
        drop=True
    )
)

icsdf_train_df.head()

,saham,bandara,marquez,motogp,marc,balapan,purbaya,set,aset,padel,...,year,dolar,kerja,level,subsidi,petenis,mereka,aceh,musim,label
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.053561,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,1
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.665318,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.083575,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,1
4,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.045516,0.0,0.0,1


In [42]:
icsdf_train_df.to_excel(
    "data_icsdf_training.xlsx",
    index=False
)

## **20. PCA**

In [43]:
pca = PCA(

    n_components=20,

    random_state=42
)

In [44]:
X_train_selected_dense = (
    X_train_selected.toarray()
)

X_test_selected_dense = (
    X_test_selected.toarray()
)

In [45]:
X_train_pca = pca.fit_transform(
    X_train_selected_dense
)

X_test_pca = pca.transform(
    X_test_selected_dense
)

In [46]:
print(
    X_train_pca.shape
)

print(
    X_test_pca.shape
)

(160, 20)
(40, 20)


## **21. Membuat tabel data reduksi training**

In [47]:
nama_pc = [
    f"PC{i}"
    for i in range(
        1,
        21
    )
]

In [48]:
df_train_reduksi = pd.DataFrame(

    X_train_pca,

    columns=nama_pc
)

df_train_reduksi["label"] = (

    y_train.reset_index(
        drop=True
    )
)

df_train_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,0.054458,-0.083656,-0.050966,-0.063402,0.049446,0.076339,-0.125293,-0.076448,-0.001681,-0.129239,...,0.059808,-0.047956,0.022836,-0.009557,-0.099971,0.032803,0.155395,-0.237321,0.365373,0
1,0.001516,-0.074960,-0.030823,-0.059307,-0.011138,0.058263,-0.016280,-0.014114,-0.009932,-0.081522,...,0.001136,0.009228,0.020869,0.006865,-0.021050,0.031943,0.095416,-0.066256,-0.004322,1
2,0.037972,-0.092182,-0.058068,-0.061232,0.042450,0.102491,-0.132471,-0.084056,-0.133353,-0.127309,...,0.236267,-0.257832,-0.049353,0.004886,0.010460,-0.035645,-0.075089,0.053086,-0.040341,0
3,-0.014009,-0.091981,-0.046848,-0.079243,0.008004,0.109202,-0.073359,-0.040299,-0.047283,-0.267736,...,-0.473267,-0.058990,-0.311360,-0.104075,0.102788,-0.072941,-0.197435,0.179236,0.086700,1
4,-0.000727,-0.103929,-0.055088,-0.111225,-0.056786,0.099267,0.055226,0.007141,0.008764,-0.088733,...,0.027889,0.052711,0.072312,0.013962,-0.096198,0.057684,0.187838,-0.356487,-0.198698,1


## **22. Data testing**

In [49]:
df_test_reduksi = pd.DataFrame(

    X_test_pca,

    columns=nama_pc
)

df_test_reduksi["label"] = (

    y_test.reset_index(
        drop=True
    )
)

df_test_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,0.025333,-0.076968,0.005153,-0.037805,0.010847,0.054619,-0.079845,-0.034686,-0.024508,-0.170149,...,-0.075585,0.103846,0.091166,-0.093016,-0.049398,0.086446,0.739421,0.503766,0.268455,0
1,-0.009996,-0.095285,0.033641,-0.025445,-0.004364,0.060042,-0.031418,-0.004604,-0.012574,-0.124149,...,0.017618,0.092739,0.225136,-0.058508,-0.015073,-0.036555,0.026122,0.045859,0.043395,1
2,-0.098914,0.041605,0.004304,-0.009282,0.005585,0.026705,-0.051870,0.047340,-0.003552,-0.039965,...,0.000947,0.015176,0.067430,-0.005413,-0.010598,-0.012220,0.015563,-0.016303,0.021734,1
3,-0.035422,-0.019355,-0.019390,-0.024950,0.001123,0.031489,-0.027728,-0.020460,-0.010153,-0.063937,...,0.015121,0.016165,0.026433,0.012956,-0.016273,0.021069,0.042084,-0.049683,-0.001718,1
4,0.140381,-0.206879,0.382122,0.168864,0.020674,-0.036795,0.013446,-0.012980,-0.014195,-0.000357,...,-0.067428,0.029574,-0.038317,-0.016361,0.005836,0.014062,0.050209,0.023424,0.013887,0


## **23. Simpan data reduksi**

In [50]:
df_train_reduksi.to_excel(
    "data_reduksi_training.xlsx",
    index=False
)

df_test_reduksi.to_excel(
    "data_reduksi_testing.xlsx",
    index=False
)

## **24. Cek total training dan testing**

In [51]:
print(
    "Training :",
    df_train_reduksi.shape
)

print(
    "Testing :",
    df_test_reduksi.shape
)

Training : (160, 21)
Testing : (40, 21)


## 26. Kolom terakhir harus label

In [52]:
df_train_reduksi.columns

Index(['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10',
       'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19',
       'PC20', 'label'],
      dtype='object')